In [1]:
# 1. Instala as ferramentas
!apt-get update && apt-get install -y ffmpeg rclone
!curl -L https://github.com/yt-dlp/yt-dlp/releases/latest/download/yt-dlp -o /usr/local/bin/yt-dlp
!chmod a+rx /usr/local/bin/yt-dlp

# 2. Escreve o script.sh diretamente no ambiente do Kaggle
with open('script.sh', 'w') as f:
    f.write(r"""#!/bin/bash
URL_DO_CANAL="${URL_DO_CANAL:-https://www.youtube.com/@republicacoisadenerd/live}"
RCLONE_CONFIG_FLAGS="--config /tmp/rclone.conf"
COOKIE_FILE="/tmp/cookies.txt"
DIRETORIO_TEMPORARIO="/tmp/gravacao"
mkdir -p "$DIRETORIO_TEMPORARIO"

echo ">>> Iniciando monitoramento (Aguardando live)..."

# Loop para esperar a live começar (Vigia por até 12 horas)
while true; do
    echo ">>> Verificando se está online: $(date)"
    
    # O yt-dlp tenta baixar. Se não houver live, ele fecha rápido.
    # Se houver live, ele grava até o fim.
    yt-dlp --cookies "$COOKIE_FILE" \
           --live-from-start \
           --merge-output-format mkv \
           -o "$DIRETORIO_TEMPORARIO/%(title)s.%(ext)s" \
           "$URL_DO_CANAL"
           
    # Verifica se algum vídeo foi criado
    VIDEO_GERADO=$(ls "$DIRETORIO_TEMPORARIO"/*.mkv 2>/dev/null)
    if [ -n "$VIDEO_GERADO" ]; then
        echo ">>> Live gravada com sucesso!"
        break
    fi
    
    echo ">>> Live ainda offline. Esperando 60 segundos..."
    sleep 60
done

# Upload para o Drive
echo ">>> Iniciando Upload para o Drive..."
rclone move "$DIRETORIO_TEMPORARIO" "MeuDrive:LivesCoisaDeNerd" $RCLONE_CONFIG_FLAGS --progress
""")

!chmod +x script.sh
print(">>> Tudo pronto!")

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:7 https://cli.github.com/packages stable/main amd64 Packages [354 B]
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:9 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,979 kB]
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:11 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [6,803 kB]
Get:12 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:13 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,959 kB]
Get:14 http://security.ubu

In [2]:
import os
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()

try:
    # Puxa os segredos (Certifique-se que os nomes no Add-ons estão EXATAMENTE assim)
    rclone_conf = user_secrets.get_secret("RCLONE_CONFIG")
    
    # Salva o config do rclone
    with open('/tmp/rclone.conf', 'w') as f:
        f.write(rclone_conf)
        
    # Tenta pegar os cookies (opcional)
    try:
        yt_cookies = user_secrets.get_secret("YOUTUBE_COOKIES")
        with open('/tmp/cookies.txt', 'w') as f:
            f.write(yt_cookies)
    except:
        print(">>> Aviso: YOUTUBE_COOKIES não encontrado. Tentando sem cookies...")
        with open('/tmp/cookies.txt', 'w') as f: f.write("")

    # Roda o script
    !bash ./script.sh

except Exception as e:
    print(f">>> ERRO CRÍTICO: {e}")

>>> Iniciando monitoramento (Aguardando live)...
>>> Verificando se está online: Sat Apr  4 11:39:41 PM UTC 2026
[youtube:tab] Extracting URL: https://www.youtube.com/@republicacoisadenerd/live
[youtube:tab] @republicacoisadenerd/live: Downloading webpage
[youtube] Extracting URL: https://www.youtube.com/watch?v=WS7-hzLUjqc
[youtube] WS7-hzLUjqc: Downloading webpage
[youtube] WS7-hzLUjqc: Downloading android vr player API JSON
[youtube] WS7-hzLUjqc: Downloading MPD manifest
[info] WS7-hzLUjqc: Downloading 1 format(s): 299+140
[dashsegments] Total fragments: unknown (live)
[download] Destination: /tmp/gravacao/ESTAMOS AO VIVO!.f299.mkv
[dashsegments] Total fragments: unknown (live)
[download] Destination: /tmp/gravacao/ESTAMOS AO VIVO!.f140.mkv






















































































































































































































































